# Plavčík a tonoucí

## Zadání slovy

> Plavčík stojí na pláži 25 m od vody. Tonoucí je 45 m stranou a 35 m od břehu.
> Po písku plavčík běží 6 m/s, ale plave jen 1,2 m/s — pětkrát pomaleji.
>
> Ve kterém místě má vběhnout do vody, aby byl u tonoucího **co nejdřív**?

Nejkratší cesta je přímka, jenže ta vede zbytečně dlouho vodou. Běžet po břehu
až naproti tonoucímu a plavat kolmo zase znamená naběhat spoustu metrů navíc.
Optimum leží mezi tím — a mnohem blíž „naproti tonoucímu“, než by člověk čekal.

## Formulace

Břeh je vodorovná osa. Plavčík stojí $A$ metrů nad ní, tonoucí $B$ metrů pod ní
a vodorovně jsou od sebe $D$. Proměnná je jediná: souřadnice $x$ místa, kde
plavčík vběhne do vody. Doba je dráha po písku dělená rychlostí běhu plus dráha
ve vodě dělená rychlostí plavání, obojí Pythagorova věta.

$$
\begin{aligned}
\text{minimize}_{x}\quad & T(x)=\frac{\lVert (x,\,A)\rVert_2}{v_1}
  +\frac{\lVert (D-x,\,B)\rVert_2}{v_2} && \text{doba do tonouciho (s)}\\
\text{subject to}\quad & 0 \le x \le D && \text{usek brehu (m)}
\end{aligned}
$$

Data: $A = 25$ m, $B = 35$ m, $D = 45$ m, $v_1 = 6$ m/s, $v_2 = 1{,}2$ m/s.

Účelová funkce je součet dvou eukleidovských norem složených s afinním
zobrazením, takže je **konvexní**: minimum je jediné a solver ho najde
z libovolného startu. CVXPY ji proto vezme přesně tak, jak je zapsaná výš.

## Od zadání ke kódu

| v zadání | v kódu |
|---|---|
| proměnná $x$ | `x = cp.Variable()` |
| $\lVert (x,\,A)\rVert_2 / v_1$ | `cp.norm(cp.hstack([x, A])) / v1` |
| $\lVert (D-x,\,B)\rVert_2 / v_2$ | `cp.norm(cp.hstack([D - x, B])) / v2` |
| $\min T(x)$ | `cp.Minimize(beh + plavani)` |
| $0 \le x \le D$ | `[x >= 0, x <= D]` |

Plná verze téhle ukázky, ze které notebook vychází, je ve skriptu [`kod/ukazka_plavcik.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C1/kod/ukazka_plavcik.py) v repozitáři předmětu.

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np

## Model a řešení

In [ ]:
# vzdálenosti v metrech, rychlosti běhu po písku a plavání v m/s
A = 25.0   # @param {type:"number"}
B = 35.0   # @param {type:"number"}
D = 45.0   # @param {type:"number"}
v1 = 6.0   # @param {type:"number"}
v2 = 1.2   # @param {type:"number"}

x = cp.Variable()                              # kde vběhnout do vody
beh = cp.norm(cp.hstack([x, A])) / v1          # dráha po písku / rychlost běhu
plavani = cp.norm(cp.hstack([D - x, B])) / v2  # dráha ve vodě / rychlost plavání
uloha = cp.Problem(cp.Minimize(beh + plavani), [x >= 0, x <= D])
# dno je ploché, tak se solveru předepíše přísnější tolerance, než má ve výchozím stavu
uloha.solve(solver=cp.CLARABEL, tol_gap_abs=1e-11, tol_gap_rel=1e-11, tol_feas=1e-11)

x_opt, t_opt = float(x.value), float(uloha.value)
print(f"úloha je konvexní podle DCP: {uloha.is_dcp()}, stav: {uloha.status}")
print(f"optimum: x* = {x_opt:.2f} m, T* = {t_opt:.2f} s")

## Kontrola, která umí selhat

Podmínka $T'(x) = 0$ se dá vyřešit tužkou a vyjde z ní **Snellův zákon**: poměr
sinů úhlů od kolmice k břehu se rovná poměru rychlostí. Solver o optice nic
neví, takže je to nezávislé měřítko — a kontrola má smysl jen tehdy, když umí
spadnout, proto ji pustíme i na řešení posunuté o metr.

In [ ]:
theta1, theta2 = np.arctan2(x_opt, A), np.arctan2(D - x_opt, B)
pomer_sinu, pomer_rychlosti = np.sin(theta1) / np.sin(theta2), v1 / v2
print(f"poměr sinů {pomer_sinu:.4f} vs. poměr rychlostí {pomer_rychlosti:.4f}")
assert abs(pomer_sinu - pomer_rychlosti) < 1e-3, "optimum neodpovídá Snellovu zákonu"

t1, t2 = np.arctan2(x_opt + 1, A), np.arctan2(D - x_opt - 1, B)
print(f"o metr vedle: poměr sinů {np.sin(t1) / np.sin(t2):.4f} — tam kontrola spadne")

## Obrázek

In [ ]:
plt.plot([0, x_opt, D], [A, 0, -B], "-o", label=f"nejrychlejší trasa, T* = {t_opt:.1f} s")
plt.axhline(0, color="gray", lw=1, label="břeh (nahoře písek, dole voda)")
plt.gca().set_aspect("equal")
plt.xlabel("vzdálenost podél břehu [m]")
plt.ylabel("vzdálenost od břehu [m]")
plt.title("Kde vběhnout do vody?")
plt.legend(loc="lower left")
plt.show()

## Na co se zeptat kódu

1. Nastavte stejné rychlosti ($v_1 = v_2$). Co udělá trasa a co poměr sinů?
2. Zpomalte plavání na $v_2 = 0{,}4$ m/s. Kterého z omezení $0 \le x \le D$ se
   řešení skoro dotkne?
3. Zapište účelovou funkci špatně — minimalizujte dráhu místo doby. Solver vrátí
   číslo i tak; jak by se na chybu přišlo bez znalosti správné odpovědi?